# Query OBIS for one species in one marine region and display summary information and map locations.

This notebook was constructed with the goal of porting functionality from an r script using "robis" OBIS_filter_by_areaid.R to python using "pyobis"

ChatGPT5.3 was utilized to help port the functionality with oversight, editing for purpose, and review by the embark project participants.

In [ ]:
"""
Query OBIS for one species in one marine region and display summary information.

Purpose:
- Retrieve OBIS occurrence records for a scientific name within an areaid
- Display summary informatio
- Map locations of the occurrence records for data exploration

Usage:
- Edit SCIENTIFIC_NAME AREA_NAME, and AREA_ID as needed

"""

import requests
import folium
from datetime import datetime
import pandas as pd
from IPython.display import display

SCIENTIFIC_NAME = "Pyrosoma atlanticum"
AREA_ID = 40003
AREA_NAME = 'California Current'

def fetch_obis_occurrences(scientific_name, area_id, page_size=1000):
    """
    Fetch all OBIS occurrence records for one scientific name and one areaid.
    Returns a pandas DataFrame.
    """
    base_url = "https://api.obis.org/v3/occurrence"
    offset = 0
    all_results = []

    while True:
        params = {
            "scientificname": scientific_name,
            "areaid": area_id,
            "size": page_size,
            "from": offset,
        }

        response = requests.get(base_url, params=params, timeout=60)
        response.raise_for_status()
        payload = response.json()

        results = payload.get("results", [])
        if not results:
            break

        all_results.extend(results)
        offset += page_size

        total = payload.get("total", 0)
        if offset >= total:
            break

    return pd.DataFrame(all_results)

records = fetch_obis_occurrences(SCIENTIFIC_NAME, AREA_ID)

## show summary info
print(f"Scientific name: {SCIENTIFIC_NAME}")
print(f"Area ID: {AREA_ID}")
print(f"Total records retrieved: {len(records)}")

Scientific name: Pyrosoma atlanticum
Area ID: 40003
Total records retrieved: 483


In [ ]:
# If you want to save as a csv file:

output_file = "obis_occurrences_pyrosoma_atlanticum_california_current.csv"
records.to_csv(output_file, index=False)
print(f"Saved to: {output_file}")

Saved to: obis_occurrences_pyrosoma_atlanticum_california_current.csv


In [ ]:
#More summary info

if records.empty:
    print("No records returned.")
else:
    print("\nColumns returned:")
    print(sorted(records.columns.tolist()))

    # Optional preview of key fields
    preview_cols = [
        "scientificName",
        "decimalLatitude",
        "decimalLongitude",
        "eventDate",
        "minimumDepthInMeters",
        "maximumDepthInMeters",
        "datasetID"
    ]

    print("\nPreview of key fields (random 10 lines)")
    display(records[preview_cols].sample(10))

    # Unique aphiaID values with WoRMS links

    aphia_values = pd.Series(records["aphiaID"].dropna().unique(), name="aphiaID")
    aphia_df = pd.DataFrame({"aphiaID": aphia_values})
    aphia_df["WoRMS_link"] = aphia_df["aphiaID"].apply(
        lambda x: f"https://marinespecies.org/aphia.php?p=taxdetails&id={int(x)}"
        if pd.notna(x) else None
    )

    display(aphia_df.sort_values("aphiaID").reset_index(drop=True))

    # Unique datasets, add OBIS links

    records["datasetURL"] = 'https://obis.org/dataset/'+ records['dataset_id'].astype(str)

    # Extract unique combinations
    datasets = (
        records[['dataset_id',"datasetName",'datasetID','datasetURL']]
        .drop_duplicates()
        .sort_values(by="datasetName")
        .reset_index(drop=True)
    )

    print(f"\nTotal unique datasets: {len(datasets)}")
    display(datasets)




Columns returned:
['absence', 'aphiaID', 'basisOfRecord', 'bathymetry', 'catalogNumber', 'class', 'classid', 'collectionCode', 'collectionID', 'continent', 'coordinateUncertaintyInMeters', 'country', 'county', 'datasetID', 'datasetName', 'dataset_id', 'dateIdentified', 'date_end', 'date_mid', 'date_start', 'date_year', 'day', 'decimalLatitude', 'decimalLongitude', 'depth', 'dropped', 'eventDate', 'eventID', 'eventRemarks', 'eventTime', 'family', 'familyid', 'fieldNumber', 'flags', 'genus', 'genusid', 'geodeticDatum', 'georeferenceProtocol', 'georeferenceRemarks', 'habitat', 'higherClassification', 'higherGeography', 'id', 'identificationRemarks', 'identifiedBy', 'individualCount', 'institutionCode', 'institutionID', 'kingdom', 'kingdomid', 'language', 'license', 'lifeStage', 'locality', 'locationID', 'locationRemarks', 'marine', 'materialSampleID', 'maximumDepthInMeters', 'maximumElevationInMeters', 'minimumDepthInMeters', 'minimumElevationInMeters', 'modified', 'month', 'node_id', 'n

,scientificName,decimalLatitude,decimalLongitude,eventDate,minimumDepthInMeters,maximumDepthInMeters,datasetID
412,Pyrosoma atlanticum,33.493200,-119.59200,2015-05-15T08:21:20Z,25.00000,30.00000,RockfishRecruitmentAndEcosystemAssessmentSurve...
457,Pyrosoma atlanticum,35.000200,-120.97130,2013-06-04T07:43:42Z,25.00000,30.00000,RockfishRecruitmentAndEcosystemAssessmentSurve...
158,Pyrosoma atlanticum,37.281700,-122.57670,2001-06-03T07:41:00Z,25.00000,30.00000,RockfishRecruitmentAndEcosystemAssessmentSurve...
268,Pyrosoma atlanticum,36.646000,-122.04720,2014-05-17T08:16:15Z,25.00000,30.00000,RockfishRecruitmentAndEcosystemAssessmentSurve...
469,Pyrosoma atlanticum,36.582000,-122.17250,2015-06-12T06:10:27Z,25.00000,30.00000,RockfishRecruitmentAndEcosystemAssessmentSurve...
19,Pyrosoma atlanticum,35.703500,-121.42880,2014-05-08T09:40:58Z,25.00000,30.00000,RockfishRecruitmentAndEcosystemAssessmentSurve...
200,Pyrosoma atlanticum,35.004700,-121.11380,2015-05-17T10:27:47Z,25.00000,30.00000,RockfishRecruitmentAndEcosystemAssessmentSurve...
85,Pyrosoma atlanticum,36.575000,-122.02830,1992-05-20T06:59:00Z,25.00000,30.00000,RockfishRecruitmentAndEcosystemAssessmentSurve...
34,Pyrosoma atlanticum,40.172904,-124.89243,2023-04-17T20:11:25Z,777.90802,777.90802,noaa-oer-okeanos-ex2301
287,Pyrosoma atlanticum,37.275000,-122.98670,1997-06-12T10:40:00Z,25.00000,30.00000,RockfishRecruitmentAndEcosystemAssessmentSurve...


,aphiaID,WoRMS_link
0,137250,https://marinespecies.org/aphia.php?p=taxdetai...



Total unique datasets: 4


,dataset_id,datasetName,datasetID,datasetURL
0,5b6251f6-a7a5-4dc9-994d-c9504b54776f,Rockfish Recruitment and Ecosystem Assessment ...,RockfishRecruitmentAndEcosystemAssessmentSurve...,https://obis.org/dataset/5b6251f6-a7a5-4dc9-99...
1,71c2c816-7e94-40b9-8e28-8172d9c5fefb,NaN,noaa-oer-okeanos-ex2301,https://obis.org/dataset/71c2c816-7e94-40b9-8e...
2,aa16d305-d413-4c4a-90be-b1ec3298d58d,NaN,NaN,https://obis.org/dataset/aa16d305-d413-4c4a-90...
3,c5687a17-e454-40f9-9a4b-d04b2c812d74,NaN,NaN,https://obis.org/dataset/c5687a17-e454-40f9-9a...


In [ ]:
# Plot OBIS occurrence records from the `records` DataFrame

def plot_obis_records(records_df):
    """
    Return a folium map of OBIS occurrence points.
    """
    required_cols = {"decimalLatitude", "decimalLongitude"}
    missing = required_cols - set(records_df.columns)

    if missing:
        raise ValueError(f"Missing required columns for mapping: {sorted(missing)}")

    map_df = records_df.dropna(subset=["decimalLatitude", "decimalLongitude"]).copy()

    if map_df.empty:
        raise ValueError("No rows with valid decimalLatitude and decimalLongitude values.")

    center_lat = map_df["decimalLatitude"].mean()
    center_lon = map_df["decimalLongitude"].mean()

    m = folium.Map(location=[center_lat, center_lon], zoom_start=4)

    for _, row in map_df.iterrows():
        popup_parts = []

        if "scientificName" in map_df.columns and pd.notna(row.get("scientificName")):
            popup_parts.append(f"scientificName: {row['scientificName']}")

        if "eventDate" in map_df.columns and pd.notna(row.get("eventDate")):
            popup_parts.append(f"eventDate: {row['eventDate']}")

        if "datasetName" in map_df.columns and pd.notna(row.get("datasetName")):
            popup_parts.append(f"datasetName: {row['datasetName']}")

        popup_text = "<br>".join(popup_parts) if popup_parts else None

        folium.CircleMarker(
            location=[row["decimalLatitude"], row["decimalLongitude"]],
            radius=3,
            weight=1,
            fill=True,
            popup=popup_text,
        ).add_to(m)

    return m

obis_map = plot_obis_records(records)
obis_map

# Source Datasets and Metadata

Dataset info pattern
https://obis.org/dataset/5b6251f6-a7a5-4dc9-994d-c9504b54776f
* You can see "Overview" and "Data Quality" sections which include information about quality flags, missing data identifiers, coordinate precision, etc.

You can get the unique set of dataset_id column in records to look at metadata for any information you need about the source datasets in obis.

In [ ]:
# We alread got the unique list of datasets from records
datasets

,dataset_id,datasetName,datasetID,datasetURL
0,5b6251f6-a7a5-4dc9-994d-c9504b54776f,Rockfish Recruitment and Ecosystem Assessment ...,RockfishRecruitmentAndEcosystemAssessmentSurve...,https://obis.org/dataset/5b6251f6-a7a5-4dc9-99...
1,71c2c816-7e94-40b9-8e28-8172d9c5fefb,NaN,noaa-oer-okeanos-ex2301,https://obis.org/dataset/71c2c816-7e94-40b9-8e...
2,aa16d305-d413-4c4a-90be-b1ec3298d58d,NaN,NaN,https://obis.org/dataset/aa16d305-d413-4c4a-90...
3,c5687a17-e454-40f9-9a4b-d04b2c812d74,NaN,NaN,https://obis.org/dataset/c5687a17-e454-40f9-9a...


In [ ]:
#You can visit the links for more info about the dataset
datasets["datasetURL"].dropna().tolist()


['https://obis.org/dataset/5b6251f6-a7a5-4dc9-994d-c9504b54776f',
 'https://obis.org/dataset/71c2c816-7e94-40b9-8e28-8172d9c5fefb',
 'https://obis.org/dataset/aa16d305-d413-4c4a-90be-b1ec3298d58d',
 'https://obis.org/dataset/c5687a17-e454-40f9-9a4b-d04b2c812d74']

In [ ]:
# Dataset citation for OBIS
# * Citation based on https://manual.obis.org/citing.html

#citaion builder based on this script's variables
now = datetime.now()
year = now.year
access_date = now.strftime("%B %d, %Y")

citation = (
    f"Ocean Biodiversity Information System (OBIS). ({year}). "
    f"Occurrence records of {SCIENTIFIC_NAME} in the {AREA_NAME} (LME {AREA_ID}). "
    f"https://obis.org (accessed {access_date})."
)

print("OBIS citation:\n")
print(citation)

OBIS citation:

Ocean Biodiversity Information System (OBIS). (2026). Occurrence records of Pyrosoma atlanticum in the California Current (LME 40003). https://obis.org (accessed May 01, 2026).
